# Fine-tune the official Deep Attention Network on FER2013

This notebook downloads the authors’ pretrained **DAN (Distract Your Attention Network)** model trained on AffectNet-8, then fine-tunes it on FER2013. It uses the [official DAN implementation](https://github.com/yaoing/DAN) and its published [AffectNet-8 checkpoint](https://drive.google.com/file/d/1uHNADViICyJEjJljv747nfvrGu12kjtu/view).

The pretrained classifier has eight AffectNet outputs. The notebook transfers the seven matching emotion rows into a FER2013 head and omits `contempt`; the FER2013 classifier is then fine-tuned on the local training split. FER2013 images are resized to 224×224 and normalized with ImageNet statistics, as in the authors’ demo.

The dataset should use image folders: `train/<emotion>/*` and `test/<emotion>/*`. The notebook keeps the official test folders out of training and uses a stratified validation split from `train`.

**Before running:** install the project requirements and `gdown` (`pip install gdown`). The first download needs internet access. Review the config cell before running the training cells; fine-tuning can take a while, especially on CPU.

In [1]:
from pathlib import Path
import json
import os
import random
import shutil
import subprocess
import sys
import urllib.request

import numpy as np
import pandas as pd
import torch
from PIL import Image
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from tqdm.auto import tqdm

print(f"PyTorch: {torch.__version__}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

PyTorch: 2.5.1+cpu
Device: cpu


## 1. Configuration

Edit the paths and training options here. The dataset root must contain `train` and `test` directories.

In [2]:
# Find the project root when this notebook is opened from the project or notebook directory.
PROJECT_ROOT = next(
    (parent for parent in [Path.cwd(), *Path.cwd().parents] if (parent / 'Datasets').is_dir()),
    Path(r'D:\FYP'),
)

CONFIG = {
    'dataset_dir': PROJECT_ROOT / 'Datasets' / 'Facial Datasets' / 'FER2013',
    'checkpoint_dir': PROJECT_ROOT / 'FER2013_Emotion_Recognition' / 'models' / 'dan_pretrained',
    'output_dir': PROJECT_ROOT / 'FER2013_Emotion_Recognition' / 'results' / 'dan_pretrained',
    'image_size': 224,
    'batch_size': 32,
    'num_workers': 0,
    'validation_fraction': 0.15,
    'seed': 42,
    'epochs': 20,
    'backbone_learning_rate': 1e-5,
    'attention_learning_rate': 1e-4,
    'classifier_learning_rate': 1e-4,
    'weight_decay': 1e-4,
    'early_stopping_patience': 5,
    'run_training': True,
}

CONFIG['dataset_dir'] = Path(CONFIG['dataset_dir']).resolve()
CONFIG['checkpoint_dir'] = Path(CONFIG['checkpoint_dir']).resolve()
CONFIG['output_dir'] = Path(CONFIG['output_dir']).resolve()
CONFIG['checkpoint_dir'].mkdir(parents=True, exist_ok=True)
CONFIG['output_dir'].mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED = CONFIG['seed']
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print('Dataset:', CONFIG['dataset_dir'])
print('Device:', DEVICE)
print('Configuration:', {k: str(v) if isinstance(v, Path) else v for k, v in CONFIG.items()})

Dataset: D:\FYP\Datasets\Facial Datasets\FER2013
Device: cpu
Configuration: {'dataset_dir': 'D:\\FYP\\Datasets\\Facial Datasets\\FER2013', 'checkpoint_dir': 'D:\\FYP\\FER2013_Emotion_Recognition\\models\\dan_pretrained', 'output_dir': 'D:\\FYP\\FER2013_Emotion_Recognition\\results\\dan_pretrained', 'image_size': 224, 'batch_size': 32, 'num_workers': 0, 'validation_fraction': 0.15, 'seed': 42, 'epochs': 20, 'backbone_learning_rate': 1e-05, 'attention_learning_rate': 0.0001, 'classifier_learning_rate': 0.0001, 'weight_decay': 0.0001, 'early_stopping_patience': 5, 'run_training': True}


## 2. Download the official DAN code and pretrained AffectNet-8 weights

The upstream checkpoint is stored on Google Drive. `gdown` retrieves it by file ID; the model source is fetched from the authors’ GitHub repository. Both files are cached under the configured checkpoint directory.

In [3]:
# Install gdown once in the active environment if needed:
# %pip install gdown
try:
    import gdown
except ImportError as exc:
    raise ImportError('Install the download helper first with: %pip install gdown') from exc

DAN_SOURCE_URL = 'https://raw.githubusercontent.com/yaoing/DAN/main/networks/dan.py'
DAN_SOURCE_PATH = CONFIG['checkpoint_dir'] / 'dan.py'
PRETRAINED_FILE_ID = '1uHNADViICyJEjJljv747nfvrGu12kjtu'
PRETRAINED_PATH = CONFIG['checkpoint_dir'] / 'affecnet8_epoch5_acc0.6209.pth'

if not DAN_SOURCE_PATH.is_file():
    urllib.request.urlretrieve(DAN_SOURCE_URL, DAN_SOURCE_PATH)
if DAN_SOURCE_PATH.stat().st_size < 1000:
    raise RuntimeError(f'Official DAN source download looks incomplete: {DAN_SOURCE_PATH}')

if not PRETRAINED_PATH.is_file() or PRETRAINED_PATH.stat().st_size < 1_000_000:
    if PRETRAINED_PATH.exists():
        PRETRAINED_PATH.unlink()
    downloaded = gdown.download(id=PRETRAINED_FILE_ID, output=str(PRETRAINED_PATH), quiet=False)
    if not downloaded or not PRETRAINED_PATH.is_file() or PRETRAINED_PATH.stat().st_size < 1_000_000:
        raise RuntimeError('The pretrained checkpoint download failed or returned an incomplete file.')

print(f'Official DAN source: {DAN_SOURCE_PATH} ({DAN_SOURCE_PATH.stat().st_size:,} bytes)')
print(f'AffectNet-8 checkpoint: {PRETRAINED_PATH} ({PRETRAINED_PATH.stat().st_size:,} bytes)')

Official DAN source: D:\FYP\FER2013_Emotion_Recognition\models\dan_pretrained\dan.py (3,494 bytes)
AffectNet-8 checkpoint: D:\FYP\FER2013_Emotion_Recognition\models\dan_pretrained\affecnet8_epoch5_acc0.6209.pth (237,019,425 bytes)


## 3. Load the pretrained network and transfer the matching emotion classes

The authors’ demo defines the AffectNet-8 order as `neutral, happy, sad, surprise, fear, disgust, anger, contempt`. FER2013 `ImageFolder` uses alphabetical class order. We copy the pretrained classifier and batch-normalization values for the seven shared classes into a seven-output head.

In [4]:
# Import the downloaded official model definition.
sys.path.insert(0, str(CONFIG['checkpoint_dir']))
from dan import DAN

AFFECTNET8_CLASSES = ['neutral', 'happy', 'sad', 'surprise', 'fear', 'disgust', 'anger', 'contempt']
FER2013_CLASSES = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
AFFECTNET8_INDEX = {name: index for index, name in enumerate(AFFECTNET8_CLASSES)}

# `pretrained=False` avoids downloading a separate backbone. The AffectNet checkpoint
# already contains the complete DAN state, including its ResNet-18 backbone.
model = DAN(num_head=4, num_class=8, pretrained=False)
try:
    checkpoint = torch.load(PRETRAINED_PATH, map_location='cpu', weights_only=True)
except Exception:
    checkpoint = torch.load(PRETRAINED_PATH, map_location='cpu', weights_only=False)
state_dict = checkpoint.get('model_state_dict', checkpoint)
model.load_state_dict(state_dict, strict=True)

old_fc = model.fc
old_bn = model.bn
model.fc = nn.Linear(old_fc.in_features, len(FER2013_CLASSES))
model.bn = nn.BatchNorm1d(len(FER2013_CLASSES))
transfer_rows = [AFFECTNET8_INDEX['anger'] if name == 'angry' else AFFECTNET8_INDEX[name] for name in FER2013_CLASSES]
with torch.no_grad():
    model.fc.weight.copy_(old_fc.weight[transfer_rows])
    model.fc.bias.copy_(old_fc.bias[transfer_rows])
    model.bn.weight.copy_(old_bn.weight[transfer_rows])
    model.bn.bias.copy_(old_bn.bias[transfer_rows])
    model.bn.running_mean.copy_(old_bn.running_mean[transfer_rows])
    model.bn.running_var.copy_(old_bn.running_var[transfer_rows])
    model.bn.num_batches_tracked.copy_(old_bn.num_batches_tracked)

model = model.to(DEVICE)
print('Loaded official DAN weights and transferred the seven FER2013 emotion outputs.')
print(f'Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

d:\FYP\.venv\Lib\site-packages\torchvision\models\_utils.py:135: UserWarning: Using 'weights' as positional parameter(s) is deprecated since 0.13 and may be removed in the future. Please use keyword parameter(s) instead.
  warnings.warn(
d:\FYP\.venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Loaded official DAN weights and transferred the seven FER2013 emotion outputs.
Trainable parameters: 19,723,733


## 4. Load FER2013 and create a stratified validation split

Train and validation images use separate transforms. The official test split is only used after fine-tuning.

In [5]:
train_dir = CONFIG['dataset_dir'] / 'train'
test_dir = CONFIG['dataset_dir'] / 'test'
if not train_dir.is_dir() or not test_dir.is_dir():
    raise FileNotFoundError(
        f'Expected image folders at {train_dir} and {test_dir}. '
        'Set CONFIG["dataset_dir"] to the FER2013 directory containing train/ and test/.'
    )

if DEVICE.type == 'cuda':
    CONFIG['batch_size'] = min(CONFIG['batch_size'], 32)

imagenet_mean = (0.485, 0.456, 0.406)
imagenet_std = (0.229, 0.224, 0.225)
train_transform = transforms.Compose([
    transforms.Resize((CONFIG['image_size'], CONFIG['image_size'])),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])
eval_transform = transforms.Compose([
    transforms.Resize((CONFIG['image_size'], CONFIG['image_size'])),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])

train_aug_dataset = datasets.ImageFolder(train_dir, transform=train_transform)
train_eval_dataset = datasets.ImageFolder(train_dir, transform=eval_transform)
test_dataset = datasets.ImageFolder(test_dir, transform=eval_transform)

if train_aug_dataset.classes != FER2013_CLASSES or test_dataset.classes != FER2013_CLASSES:
    raise ValueError(
        f'Expected FER2013 classes {FER2013_CLASSES}; found train={train_aug_dataset.classes}, '
        f'test={test_dataset.classes}'
    )
labels = np.asarray(train_aug_dataset.targets)
train_indices, val_indices = train_test_split(
    np.arange(len(labels)),
    test_size=CONFIG['validation_fraction'],
    random_state=SEED,
    stratify=labels,
)
train_subset = Subset(train_aug_dataset, train_indices.tolist())
val_subset = Subset(train_eval_dataset, val_indices.tolist())

loader_options = {
    'batch_size': CONFIG['batch_size'],
    'num_workers': CONFIG['num_workers'],
    'pin_memory': DEVICE.type == 'cuda',
}
train_loader = DataLoader(train_subset, shuffle=True, **loader_options)
val_loader = DataLoader(val_subset, shuffle=False, **loader_options)
test_loader = DataLoader(test_dataset, shuffle=False, **loader_options)

print(f'Classes: {train_aug_dataset.classes}')
print(f'Train: {len(train_subset):,} | Validation: {len(val_subset):,} | Test: {len(test_dataset):,}')

Classes: ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
Train: 24,402 | Validation: 4,307 | Test: 7,178


## 5. Fine-tune DAN on FER2013

The full pretrained model is trainable. The backbone uses a smaller learning rate than the attention modules and classifier. The best validation-accuracy checkpoint is saved for final test evaluation.

In [6]:
# Separate parameter groups so the pretrained backbone changes more gradually.
backbone_params = list(model.features.parameters())
attention_params = [
    parameter for name, parameter in model.named_parameters()
    if name.startswith('cat_head')
]
classifier_params = list(model.fc.parameters()) + list(model.bn.parameters())
optimizer = torch.optim.AdamW(
    [
        {'params': backbone_params, 'lr': CONFIG['backbone_learning_rate']},
        {'params': attention_params, 'lr': CONFIG['attention_learning_rate']},
        {'params': classifier_params, 'lr': CONFIG['classifier_learning_rate']},
    ],
    weight_decay=CONFIG['weight_decay'],
)
criterion = nn.CrossEntropyLoss()
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=2, factor=0.3)
BEST_CHECKPOINT = CONFIG['checkpoint_dir'] / 'dan_fer2013_finetuned_best.pt'
HISTORY_PATH = CONFIG['output_dir'] / 'training_history.json'

@torch.no_grad()
def predict_loader(loader):
    model.eval()
    actual, predicted = [], []
    for images, labels in loader:
        logits = model(images.to(DEVICE))[0]
        predicted.extend(logits.argmax(dim=1).cpu().tolist())
        actual.extend(labels.tolist())
    return actual, predicted

history = []
best_val_accuracy = -1.0
stale_epochs = 0

if CONFIG['run_training']:
    for epoch in range(1, CONFIG['epochs'] + 1):
        model.train()
        running_loss, seen = 0.0, 0
        progress = tqdm(train_loader, desc=f'Epoch {epoch}/{CONFIG["epochs"]} train')
        for images, labels in progress:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            logits = model(images)[0]
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * labels.size(0)
            seen += labels.size(0)
            progress.set_postfix(loss=f'{running_loss / max(seen, 1):.4f}')

        val_actual, val_predicted = predict_loader(val_loader)
        val_accuracy = accuracy_score(val_actual, val_predicted)
        val_macro_f1 = f1_score(val_actual, val_predicted, average='macro', zero_division=0)
        record = {
            'epoch': epoch,
            'train_loss': running_loss / max(seen, 1),
            'validation_accuracy': val_accuracy,
            'validation_macro_f1': val_macro_f1,
            'learning_rates': [group['lr'] for group in optimizer.param_groups],
        }
        history.append(record)
        print(record)
        scheduler.step(val_accuracy)

        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy
            stale_epochs = 0
            torch.save({
                'model_state_dict': model.state_dict(),
                'classes': FER2013_CLASSES,
                'pretrained_source': 'DAN AffectNet-8 official checkpoint',
                'config': {key: str(value) if isinstance(value, Path) else value for key, value in CONFIG.items()},
                'epoch': epoch,
                'validation_accuracy': val_accuracy,
                'validation_macro_f1': val_macro_f1,
            }, BEST_CHECKPOINT)
        else:
            stale_epochs += 1
            if stale_epochs >= CONFIG['early_stopping_patience']:
                print('Early stopping: validation accuracy did not improve.')
                break

    HISTORY_PATH.write_text(json.dumps(history, indent=2), encoding='utf-8')
    print(f'Best fine-tuned checkpoint: {BEST_CHECKPOINT}')
else:
    print('Training is disabled. Set CONFIG["run_training"] = True to fine-tune.')

Epoch 1/20 train:   0%|          | 0/763 [00:00<?, ?it/s]

{'epoch': 1, 'train_loss': 1.1284549416953271, 'validation_accuracy': 0.6654283724169956, 'validation_macro_f1': 0.6228320354390996, 'learning_rates': [1e-05, 0.0001, 0.0001]}


Epoch 2/20 train:   0%|          | 0/763 [00:00<?, ?it/s]

{'epoch': 2, 'train_loss': 1.0347546295361112, 'validation_accuracy': 0.6772695611794752, 'validation_macro_f1': 0.6362317348328858, 'learning_rates': [1e-05, 0.0001, 0.0001]}


Epoch 3/20 train:   0%|          | 0/763 [00:00<?, ?it/s]

{'epoch': 3, 'train_loss': 0.9806974373697385, 'validation_accuracy': 0.6814488042721152, 'validation_macro_f1': 0.6438597290465703, 'learning_rates': [1e-05, 0.0001, 0.0001]}


... (18 outputs left out)

Epoch 13/20 train:   0%|          | 0/763 [00:00<?, ?it/s]

{'epoch': 13, 'train_loss': 0.600083953796298, 'validation_accuracy': 0.698397956814488, 'validation_macro_f1': 0.6705680904479824, 'learning_rates': [3e-06, 3e-05, 3e-05]}


Epoch 14/20 train:   0%|          | 0/763 [00:00<?, ?it/s]

{'epoch': 14, 'train_loss': 0.5834047383934189, 'validation_accuracy': 0.6986301369863014, 'validation_macro_f1': 0.6724657852562208, 'learning_rates': [3e-06, 3e-05, 3e-05]}
Early stopping: validation accuracy did not improve.
Best fine-tuned checkpoint: D:\FYP\FER2013_Emotion_Recognition\models\dan_pretrained\dan_fer2013_finetuned_best.pt


## 6. Evaluate the best checkpoint on the official test split

In [7]:
if BEST_CHECKPOINT.is_file():
    try:
        saved = torch.load(BEST_CHECKPOINT, map_location=DEVICE, weights_only=True)
    except Exception:
        saved = torch.load(BEST_CHECKPOINT, map_location=DEVICE, weights_only=False)
    model.load_state_dict(saved['model_state_dict'], strict=True)
    y_true, y_pred = predict_loader(test_loader)

    test_accuracy = accuracy_score(y_true, y_pred)
    test_macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    report = classification_report(
        y_true, y_pred, labels=list(range(len(FER2013_CLASSES))),
        target_names=FER2013_CLASSES, output_dict=True, zero_division=0,
    )
    metrics = {
        'test_accuracy': test_accuracy,
        'test_macro_f1': test_macro_f1,
        'classification_report': report,
        'confusion_matrix': confusion_matrix(y_true, y_pred, labels=list(range(len(FER2013_CLASSES)))).tolist(),
        'checkpoint': str(BEST_CHECKPOINT),
    }
    (CONFIG['output_dir'] / 'test_metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
    print(f'Test accuracy: {test_accuracy:.4f}')
    print(f'Test macro F1: {test_macro_f1:.4f}')
    display(pd.DataFrame(report).T.loc[FER2013_CLASSES])
else:
    print('No fine-tuned checkpoint found. Run the training cell first.')

Test accuracy: 0.6838
Test macro F1: 0.6447


,precision,recall,f1-score,support
angry,0.597378,0.665971,0.629812,958.0
disgust,0.528302,0.504505,0.516129,111.0
fear,0.533333,0.421875,0.471101,1024.0
happy,0.872867,0.894025,0.883319,1774.0
neutral,0.635409,0.686942,0.660171,1233.0
sad,0.571978,0.595830,0.583661,1247.0
surprise,0.813423,0.729242,0.769036,831.0


## References

- Authors’ [official DAN code and model links](https://github.com/yaoing/DAN). The repository lists AffectNet-8 and RAF-DB pretrained checkpoints; this notebook uses AffectNet-8 because its label set overlaps with all seven FER2013 classes.
- Wang et al., [Distract Your Attention: Multi-Head Cross Attention Network for Facial Expression Recognition](https://arxiv.org/abs/2109.07270).
- The downloaded source and weights remain under `models/dan_pretrained`; fine-tuned weights and metrics are written under `models/dan_pretrained` and `results/dan_pretrained`.